In [1]:
import modules.data as d
import modules.model as m
import modules.train as t
import modules.utils as u

import torch
import torch.nn as nn

from pathlib import Path

In [2]:
# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device()

# get data
brca = d.Data(
    # datasets
    tcga_project = 'TCGA-BRCA',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',

    # dirs
    tcga_dir = datasets / 'tcga',
    relation_filepath = datasets / 'other/relation_ohe.csv',
    
    # col, filter
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Metastatic']},
    max_subset=120,
)

*** Device() ***
device = cuda:2

**** Data() ****
log0_method      str
gene_counts      (4383, 567)      DataFrame
metadata         (567, 3)         DataFrame
relation         (75939, 19)      DataFrame
node_id_map      dict
masks            305              list
X                (567, 4383, 1)   Tensor (cuda:2)
y                (567, 6)         Tensor (cuda:2)
y_labels         6                list
num_samples      567              int
num_nodes        4383             int
num_features     1                int
num_labels       6                int
num_masks        305              int



In [ ]:
# initialize experiment
brca_GCNc = t.Experiment(
    data=brca,
    generator=generator,
)

# run experiment
brca_GCNc.run_experiment(
    model_class=m.GCNClassifier,
    model_kwargs={
        'in_features':brca.num_features,
        'out_features':brca.num_labels,
        'relation':brca.relation,
        'num_nodes':brca.num_nodes,
        'gcn_kwargs':{'end_fn':nn.LeakyReLU()}
    },

    training_class=t.ClassifierTrainingModule,
    training_kwargs={
        'loss_fn':nn.CrossEntropyLoss(brca_GCNc.class_weights),
        'optimizer':torch.optim.Adam,
        'num_epochs':100,
        'report_metrics':['tot_loss', 'accuracy'],
    },

    num_trials=10,
    comment='experiment_class_test'
)



100%|██████████| 100/100 [00:03<00:00, 27.30it/s, Epoch 99      Train: tot_loss=4.4262    accuracy=0.8942        Val: tot_loss=3.4729    accuracy=0.8824]


Test	 tot_loss=6.9109    accuracy=0.8588



100%|██████████| 100/100 [00:03<00:00, 25.75it/s, Epoch 99      Train: tot_loss=0.1596    accuracy=0.9975        Val: tot_loss=2.4947    accuracy=0.9294]


Test	 tot_loss=6.9879    accuracy=0.8353



 46%|████▌     | 46/100 [00:01<00:01, 28.63it/s, Epoch 46      Train: tot_loss=6.7118    accuracy=0.8589        Val: tot_loss=6.4344    accuracy=0.8235] 